# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [2]:
!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [3]:
import pandas as pd
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import re
import gc

In [4]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        if key == "pedido_sugerido/data-v1/cam/ventas_sellout_cam000":
            print(f"OJO: El key {key} aplica a la validacion.")
            continue
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [5]:
comprobar_inputs()

pedido_sugerido/data-v1/cam/
pedido_sugerido/data-v1/cam/D_stock_cr.csv
pedido_sugerido/data-v1/cam/D_stock_gt.csv
pedido_sugerido/data-v1/cam/D_stock_ni.csv
pedido_sugerido/data-v1/cam/D_stock_pa.csv
pedido_sugerido/data-v1/cam/maestro_productos_cam000
pedido_sugerido/data-v1/cam/ventas_cam000
pedido_sugerido/data-v1/cam/ventas_cam001
OJO: El key pedido_sugerido/data-v1/cam/ventas_sellout_cam000 aplica a la validacion.
pedido_sugerido/data-v1/cam/visitas_cam000
Todo bien :D


In [6]:
# Obtener cod SKUs y descripcion de CAM
query=f'''select * from
    comercial_cam.dim_producto
    where estado='A' and cod_compania = '0081';
    '''

con = wr.data_api.redshift.connect(cluster_id = 'dwh-cloud-storage-salesforce-prod', database = 'dwh_prod',db_user='dwhuser')
maestro_prod = wr.data_api.rds.read_sql_query( query, con)

In [7]:
maestro_prod = maestro_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
maestro_prod = maestro_prod.groupby(["cod_articulo_magic"]).first().reset_index()
maestro_prod.to_csv("Input/D_pan_masterProd.csv", index=False)

In [8]:
# maestro_prod.columns

In [9]:
# maestro_prod[~maestro_prod.desc_unidad_negocio.isnull()]

In [10]:
# maestro_prod[["cod_articulo_magic","desc_articulo"]].drop_duplicates().reset_index(drop=True)

In [11]:
rutas_test = [
# Piloto    
34278,34221,34209,34221,34222,34223,34224,34225,34226,34272,34273,34227,34228,34229,34230,
34231,34232,34233,34234,34235,34281,34236,34237,34238,34239,34240,34241,34242,34243,34244,
34282,34245,34246,34247,34248,34249,34250,34251,34252,34253,34286,34254,34255,34256,34257,
34258,34259,34260,34261,34262,34287,34263,34264,34265,34266,34267,34268,34269,34270,34271,
34285,34274,34275,34276,34277,34278,34279,34209,34208,34211,34213,34212,34218,34214,34217,
34219,34216,34215]

In [12]:
len(rutas_test)

78

In [13]:
rutas_test = list(set(rutas_test)) # quitamos rutas duplicadas
len(rutas_test)

75

In [14]:
clientes_ruta_test = []

In [15]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_cam000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df.pais__c == "NI")
                    & (df["cod_ruta"].isin(rutas_test))
                    & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [16]:
%%time
descargar_visitas()

{'ResponseMetadata': {'RequestId': 'JRZFHCD3C5QC2WS9', 'HostId': '900bOk0npFjVtR/zdib8D007oK+V4jYLzKpnpQ5GOwZlbaIA9AyP5lzf88PdSBUR0NByWzkPiUA=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '900bOk0npFjVtR/zdib8D007oK+V4jYLzKpnpQ5GOwZlbaIA9AyP5lzf88PdSBUR0NByWzkPiUA=', 'x-amz-request-id': 'JRZFHCD3C5QC2WS9', 'date': 'Mon, 30 Mar 2026 21:40:56 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 37, tzinfo=tzlocal()), 'ETag': '"7dda2ed19935555f934134f38e064ffd"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': 'FULL_OBJECT', 'Size': 41184, 'S

In [17]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Creamos DF que juntará los 2 archivos
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0:
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_cam001":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)

            else:
                continue

    nombre_csv = f"Input/ventas_cam_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [18]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'W0AV1CNSBQG9QTPQ', 'HostId': 'qV7pCFXNXS2J8uIWPAG04jG7SGZPyxh06q+aS8tj7MdMqJ984twhJ4cLeXWGDPAmfgW+njq9EiTyN98Ft6cSiDGvoIOJDK2V', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'qV7pCFXNXS2J8uIWPAG04jG7SGZPyxh06q+aS8tj7MdMqJ984twhJ4cLeXWGDPAmfgW+njq9EiTyN98Ft6cSiDGvoIOJDK2V', 'x-amz-request-id': 'W0AV1CNSBQG9QTPQ', 'date': 'Mon, 30 Mar 2026 21:40:58 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 37, tzinfo=tzlocal()), 'ETag': '"7dda2ed19935555f934134f38e064ffd"', 'ChecksumAlgorithm': ['CRC32'], 'Checksu

/tmp/ipykernel_30324/2670914792.py:28: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(content), sep=";")


CPU times: user 2min 6s, sys: 25.7 s, total: 2min 32s
Wall time: 4min 24s


## 1.1 Juntar VENTAS y VISITAS

In [19]:
import pandas as pd
from datetime import datetime, timedelta

In [20]:
pan_ventas = pd.read_parquet("Input/ventas_cam_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_cam000.parquet")

In [21]:
pan_ventas = pan_ventas[pan_ventas.cod_pais=="NI"].reset_index(drop=True)

In [22]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c==2].reset_index(drop=True)

In [23]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [24]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260330


In [25]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-30'

In [26]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Nicaragua/Input/visitas_cam000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Nicaragua/Input/visitas_cam000_2026-03-30.csv'],
 'partitions_values': {}}

In [27]:
PS_VAL_total_skus_venta = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta

87

In [28]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str((x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [29]:
PS_VAL_ultima_venta = pan_ventas["fecha_liquidacion"].sort_values().unique()[-1:]
PS_VAL_ultima_venta

array(['2026-03-29'], dtype=object)

In [30]:
PS_VAL_ultima_visita = pan_visitas["ultima_visita"].sort_values().unique()[-2:-1]
PS_VAL_ultima_visita

array(['2026-03-28'], dtype=object)

In [31]:
PS_VAL_total_clientes_venta = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_venta

13270

In [32]:
PS_VAL_total_clientes_visita = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita

12994

In [33]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c == 2].reset_index(drop=True)

In [34]:
PS_VAL_total_clientes_visita_trad = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita_trad

12994

In [35]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [36]:
# Quitamos cod_modulo de ventas para luego quedarnos con el de visitas
# pan_ventas = pan_ventas.drop(["cod_modulo","cod_ruta"], axis=1)

In [37]:
# pan_ventas=pd.merge(pan_ventas,pan_visitas[["id_cliente","dias_de_visita__c","periodo_de_visita__c","ultima_visita","cod_ruta","cod_modulo"]],on=["id_cliente"],how="outer").drop_duplicates()

In [38]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [39]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

1347

In [40]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [41]:
# Al cruzar visita con ventas. hay clientes de. visita que no tienen informacion de venta, esto se remplaza a nivel de ruta.
# si la ruta entera no tiene ventas, se ignora
# Este df guarda clientes con visitas y sin ventas para luego recomendarlos por separado
df_clientes_con_visitas_sin_ventas = pd.DataFrame()
for ruta in pan_ventas["cod_ruta"].unique():
    temp_df = pan_ventas[pan_ventas["cod_ruta"] == ruta]
    # Si la ruta entera no tiene ninguna venta, se guarda en un dataframe aparte
    if len(temp_df) == (temp_df["cod_articulo_magic"].isnull().sum()):
        df_clientes_con_visitas_sin_ventas = pd.concat(
            [df_clientes_con_visitas_sin_ventas, temp_df], axis=0
        ).reset_index(drop=True)

In [42]:
pan_ventas["id_cliente"].nunique()

12948

In [43]:
PS_VAL_total_clientes_merged = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_merged

12948

In [44]:
PS_VAL_total_skus_merged = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_merged

87

In [45]:
PS_VAL_total_fechas_unicas = pan_ventas.fecha_liquidacion.sort_values().nunique()
PS_VAL_total_fechas_unicas

332

In [46]:
pan_ventas.groupby("cod_sucursal").agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
)

,cod_articulo_magic,id_cliente,cod_ruta
cod_sucursal,,,
2,87,10397,64
5,87,2551,11


# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

In [47]:
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/D_pan_masterProd.csv")

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [48]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"}, inplace=True)
pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [49]:
pan_ventas.to_parquet("Processed/ventas_12m.parquet", index=False)

## 2.3 Seleccionando SKUs que no se commpran en los ultimos 3 dias
La variable **sucursal_sku** contiene todos los SKUs disponibles por sucursal

In [50]:
# pan_ventas[["cod_compania","cod_sucursal"]].drop_duplicates().to_csv("NI_compa_sucursal_2025-08-12.csv",index=False)

In [51]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [52]:
sucursales = pan_ventas["cod_sucursal"].unique()
sucursales

array([2, 5])

In [53]:
# Obtener la fecha actual
fecha_actual = datetime.now(pytz.timezone("America/Lima")).date()
# Lista para almacenar las fechas
last_3_days = []
dias_atras = 3
sucursal_sku = {}
# Obtener las fechas de los últimos 3 días
for i in range(1, dias_atras + 1):
    fecha = fecha_actual - timedelta(days=i)
    fecha_formato = fecha.strftime("%Y-%m-%d")
    last_3_days.append(fecha_formato)

In [54]:
last_3_days

['2026-03-29', '2026-03-28', '2026-03-27']

In [55]:
for sucursal in sucursales:
    temp = pan_ventas[pan_ventas["cod_sucursal"] == sucursal]
    # sku_fecha=temp[["cod_articulo_magic","fecha_liquidacion"]].sort_values(["cod_articulo_magic","fecha_liquidacion"]).drop_duplicates().reset_index(drop=True)
    # sku_fecha=sku_fecha.groupby("cod_articulo_magic")[["cod_articulo_magic","fecha_liquidacion"]].tail(3)
    # #Listando SKUs comprados al menos una vez en los ultimos 3 dias
    # sku_in_last_3_days=sku_fecha[sku_fecha.fecha_liquidacion.isin(last_3_days)]["cod_articulo_magic"].unique()
    # print(f"Sucursal {sucursal} eliminando SKUs:",sku_fecha[(~sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days))].reset_index(drop=True)["cod_articulo_magic"].sort_values().unique())
    # #SKU COMPRADO AL MENOS una vez en los ultimos 3 dias
    # sku_fecha=sku_fecha[sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days)].reset_index(drop=True)
    # temp=temp[temp["cod_articulo_magic"].isin(sku_fecha["cod_articulo_magic"].unique())].reset_index(drop=True)
    sucursal_sku[sucursal] = temp["cod_articulo_magic"].sort_values().unique()

## 2.4 Filtrar clientes a visitar mañana

In [56]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [57]:
# Filtrar ventas mayores a 0
# pan_ventas=pan_ventas[pan_ventas["imp_netovta"]>0]
# En lugar de omitir estos clientes, mejor los consideramos como OPTIMIZAR
# pan_ventas.loc[pan_ventas["imp_netovta"] <= 0, "new_segment"] = "OPTIMIZAR"

In [58]:
PS_VAL_total_clientes_imp_netovta_positivo = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_imp_netovta_positivo

12948

In [59]:
pan_ventas.dias_de_visita__c.unique()

array(['2;5', '1;4', '3;6', '2'], dtype=object)

In [60]:
pan_ventas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1;4    4255
2         1
2;5    4257
3;6    4435
Name: id_cliente, dtype: int64

In [61]:
pan_ventas.groupby("periodo_de_visita__c").id_cliente.nunique()

periodo_de_visita__c
F1    12948
Name: id_cliente, dtype: int64

In [62]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 30, 21, 46, 18, 451127)

In [63]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 1
dia_siguiente 2


In [64]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [65]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [66]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [67]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['2;5', '2'], dtype=object)

In [68]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)

In [69]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [70]:
pan_ventas.dias_de_visita__c.unique()

array(['2;5', '2'], dtype=object)

In [71]:
PS_VAL_total_clientes_para_manhana = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_para_manhana

4258

In [72]:
PS_VAL_total_skus_venta_para_manhana = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta_para_manhana

87

In [73]:
pan_ventas.groupby("cod_sucursal").cod_articulo_magic.nunique()

cod_sucursal
2    87
5    87
Name: cod_articulo_magic, dtype: int64

In [74]:
pan_ventas.to_parquet("Processed/ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [75]:
df_ventas = pd.read_parquet("Processed/ventas_manana.parquet")

In [76]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [77]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [78]:
df_ventas.desc_marca.unique()

array(['BIG', 'CIFRUT', 'PULP', 'CIELO', 'AJE CIELO', 'SPORADE', 'DILYTE',
       'D GUSSTO', 'BIO ALOE', 'VOLT', 'VOLT GO'], dtype=object)

In [79]:
df_ventas.desc_categoria.unique()

array(['GASEOSAS', 'JUGOS LIGEROS', 'NECTAR', 'AGUA',
       'BEBIDAS DEPORTIVAS', 'SUERO', 'SOPAS', 'ALOE VERA', 'SNACKS',
       'CEREALES', 'ENERGIZANTE', 'HARINAS'], dtype=object)

In [80]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()
    print(ranks.set_index("index")["Ranking"].to_dict())

********************
GIRO:  VENTA EN CASA PARTICULAR
Categorias:  ['GASEOSAS', 'JUGOS LIGEROS', 'NECTAR', 'AGUA', 'ENERGIZANTE', 'CEREALES', 'BEBIDAS DEPORTIVAS', 'ALOE VERA', 'SNACKS', 'SOPAS', 'SUERO', 'HARINAS']
Counts:  [886700.6666666666, 142747.0833333333, 107854.0, 28673.749999999996, 13514.58333333333, 13036.0, 12888.49999999999, 8433.16666666666, 5211.0, 2333.0, 1474.6666666666658, 142.8888888888794]
{'GASEOSAS': 1, 'JUGOS LIGEROS': 1, 'NECTAR': 2, 'AGUA': 2, 'ENERGIZANTE': 7, 'CEREALES': 8, 'BEBIDAS DEPORTIVAS': 9, 'ALOE VERA': 10, 'SNACKS': 11, 'SOPAS': 12, 'SUERO': 13, 'HARINAS': 14}
********************
GIRO:  RESTAURANT TRADICIONAL
Categorias:  ['GASEOSAS', 'JUGOS LIGEROS', 'AGUA', 'ENERGIZANTE', 'NECTAR', 'BEBIDAS DEPORTIVAS', 'ALOE VERA', 'SOPAS', 'SUERO', 'CEREALES', 'SNACKS', 'HARINAS']
Counts:  [5072.0, 803.3333333333333, 397.0, 326.5, 319.0, 180.0, 88.0, 23.0, 20.5, 18.0, 16.0, 3.5555555555555]
{'GASEOSAS': 1, 'JUGOS LIGEROS': 1, 'AGUA': 2, 'ENERGIZANTE': 2, 'NECTAR

## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [81]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)
rutas

array([34218, 34219, 34212, 34208, 34211, 34213, 34215, 34214, 34216,
       34274, 34272, 34270, 34209, 34217, 34247, 34278, 34232, 34251,
       34238, 34277, 34279, 34225, 34257, 34260, 34240, 34266, 34262,
       34276, 34265, 34253, 34222, 34275, 34286, 34236, 34246, 34252,
       34250, 34255, 34241, 34230, 34249, 34245, 34239, 34229, 34226,
       34267, 34263, 34243, 34256, 34285, 34281, 34273, 34287, 34235,
       34271, 34227, 34237, 34233, 34254, 34224, 34259, 34242, 34282,
       34244, 34223, 34248, 34258, 34269, 34264, 34268, 34228, 34221,
       34234, 34261, 34231])

In [82]:
low_sku_ruta = []

**Para rutas que tengan más de 45 SKUS**

In [83]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() <= 5:
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 34218
SKUs disponibles: 86
*********************
Ruta: 34219
SKUs disponibles: 85
*********************
Ruta: 34212
SKUs disponibles: 83
*********************
Ruta: 34208
SKUs disponibles: 84
*********************
Ruta: 34211
SKUs disponibles: 86
*********************
Ruta: 34213
SKUs disponibles: 86
*********************
Ruta: 34215
SKUs disponibles: 86
*********************
Ruta: 34214
SKUs disponibles: 87
*********************
Ruta: 34216
SKUs disponibles: 85
*********************
Ruta: 34274
SKUs disponibles: 84
*********************
Ruta: 34272
SKUs disponibles: 86
*********************
Ruta: 34270
SKUs disponibles: 87
*********************
Ruta: 34209
SKUs disponibles: 84
*********************
Ruta: 34217
SKUs disponibles: 85
*********************
Ruta: 34247
SKUs disponibles: 82
*********************
Ruta: 34278
SKUs disponibles: 86
*********************
Ruta: 34232
SKUs disponibles: 84
*********************
Ruta: 34251
SKUs disponibles: 76
**********

**Para rutas que tengan menos de 45 SKUS**

In [84]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: []
SKUs disponibles: 0


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [85]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [86]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [87]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [88]:
# Se generan csv en Output/Panama/4-resultados/
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    als_records = als_records.withColumn(
        "clienteId", concat("cod_compania", "cod_cliente")
    )
    als_records = als_records.withColumn(
        "clienteId", col("clienteId").cast(IntegerType())
    )
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(subset=["clienteId", "cod_articulo_magic"])

    input_cols = ["ClienteId", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als = ALS(
        rank=15,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="ClienteId",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    recs = recs.select("ClienteId", "recommendations.cod_articulo_magic")
    recs_to_parse = recs.toPandas()
    # print(recs_to_parse)

    lista_rec = [f"r{i+1}" for i in range(sku_len)]

    recs_to_parse[lista_rec] = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(), index=recs_to_parse.index
    )
    client_recs = pd.melt(
        recs_to_parse, id_vars=["ClienteId"], value_vars=lista_rec
    )

    client_recs["compania"] = client_recs["ClienteId"].apply(lambda x: str(x)[:2])
    client_recs["cliente"] = client_recs["ClienteId"].apply(lambda x: str(x)[2:])
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

    # client_recs.to_csv("Output/Panama/rutas/D_rutas_rec.csv", mode='a', header=False, index=False,sep=";")

## 4.3 Corriendo PS

In [89]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [90]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([34218, 34219, 34212, 34208, 34211, 34213, 34215, 34214, 34216,
       34274, 34272, 34270, 34209, 34217, 34247, 34278, 34232, 34251,
       34238, 34277, 34279, 34225, 34257, 34260, 34240, 34266, 34262,
       34276, 34265, 34253, 34222, 34275, 34286, 34236, 34246, 34252,
       34250, 34255, 34241, 34230, 34249, 34245, 34239, 34229, 34226,
       34267, 34263, 34243, 34256, 34285, 34281, 34273, 34287, 34235,
       34271, 34227, 34237, 34233, 34254, 34224, 34259, 34242, 34282,
       34244, 34223, 34248, 34258, 34269, 34264, 34268, 34228, 34221,
       34234, 34261, 34231])

In [91]:
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 34218: 86


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 21:46:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 21:46:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/30 21:47:01 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/30 21:47:01 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/30 21:47:01 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 34219: 85


*********************
SKUs disponibles en ruta 34212: 83
*********************
SKUs disponibles en ruta 34208: 84
*********************
SKUs disponibles en ruta 34211: 86
*********************
SKUs disponibles en ruta 34213: 86
*********************
SKUs disponibles en ruta 34215: 86
*********************
SKUs disponibles en ruta 34214: 87
*********************
SKUs disponibles en ruta 34216: 85
*********************
SKUs disponibles en ruta 34274: 84
*********************
SKUs disponibles en ruta 34272: 86
*********************
SKUs disponibles en ruta 34270: 87
*********************
SKUs disponibles en ruta 34209: 84
*********************
SKUs disponibles en ruta 34217: 85
*********************
SKUs disponibles en ruta 34247: 82
*********************
SKUs disponibles en ruta 34278: 86
*********************
SKUs disponibles en ruta 34232: 84
*********************
SKUs disponibles en ruta 34251: 76
*********************
SKUs disponibles en ruta 34238: 80
*********************
SKUs disp

# 5. Ajustes al output del algoritmo ALS

## 5.1 Se usa el archivo maestro para no recomendar SKUs que no se deben a las rutas

In [92]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [93]:
pan_rec.shape

(353822, 2)

In [94]:
pan_rec.id_cliente.nunique()

4258

In [95]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [96]:
pan_rec.isnull().sum()

id_cliente            0
cod_articulo_magic    0
cod_ruta              0
dtype: int64

In [97]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

83.09581963363081

In [98]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/maestro_productos_cam000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_cam_20250217.csv",sep=";")
# skus_val.head()

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [99]:
skus_val = skus_val[skus_val.cod_compania==81]
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "CAM"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

fecha_proceso:  20260330


In [100]:
skus_val.shape

(452272, 14)

In [101]:
# skus_val = pd.read_csv("productos_pedido_sugerido_validacion_cam_090225.csv",sep=";")
# skus_val = skus_val[skus_val.cod_compania==70].reset_index(drop=True)

In [102]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

1.0

In [103]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [104]:
pan_rec.shape

(142732, 3)

In [105]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

33.52090183184594

In [106]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [107]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0081|120250,523192
1,CAM|0081|121190,500094
2,CAM|0081|118300,517499
3,CAM|0081|118120,517507
4,CAM|0081|119020,509552


In [108]:
pan_rec.shape

(142732, 2)

In [109]:
pan_rec.id_cliente.nunique()

4258

In [110]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.2 Se quita los SKUs de las ultimas 2 semanas (Evitar recompra)

### 5.1. Obteniendo marcas de recomendaciones por cliente

In [111]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [112]:
df_ventas = pd.read_parquet("Processed/ventas_manana.parquet")

In [113]:
pan_rec.id_cliente.nunique()

4258

In [114]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [115]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(87, 2)


,desc_categoria,cod_articulo_magic
0,GASEOSAS,500138
1,JUGOS LIGEROS,509906
3,NECTAR,598821
4,GASEOSAS,500032
5,GASEOSAS,522741


In [116]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(142732, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0081|120250,523192,CEREALES
1,CAM|0081|121190,500094,GASEOSAS
2,CAM|0081|118300,517499,ALOE VERA
3,CAM|0081|118120,517507,ALOE VERA
4,CAM|0081|119020,509552,JUGOS LIGEROS
...,...,...,...
142727,CAM|0081|93023,522501,BEBIDAS DEPORTIVAS
142728,CAM|0081|101205,621967,HARINAS
142729,CAM|0081|101205,523648,SNACKS
142730,CAM|0081|101189,511908,JUGOS LIGEROS


### 5.2. Calcular categorias distintas recomendadas por cliente

In [117]:
# Conteo de categorias recomendadas por Cliente
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(4258, 2)


,id_cliente,desc_categoria
0,CAM|0081|100001,5
1,CAM|0081|100005,6
2,CAM|0081|100008,6
3,CAM|0081|100011,5
4,CAM|0081|100013,9


### 5.3 Se quita los SKUs de las ultimas 2 semanas (Evitar recompra)

In [118]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

Timestamp('2026-03-16 16:51:31.628754-0500', tz='America/Lima')

In [119]:
pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()

33.52090183184594

In [120]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT")

In [121]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [122]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [123]:
# Clientes iniciales
cliente_rec_marca.id_cliente.nunique()

4258

In [124]:
# Clientes luego de quitar skus de las ultimas 2 semanas
cliente_rec_sin4.id_cliente.nunique()

4240

In [125]:
pan_rec = cliente_rec_sin4[["id_cliente","cod_articulo_magic"]]

In [126]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0081|120250,523192
1,CAM|0081|121190,500094
2,CAM|0081|118300,517499
3,CAM|0081|118120,517507
4,CAM|0081|119020,509552


In [127]:
pan_rec.shape

(111968, 2)

In [128]:
pan_rec.id_cliente.nunique()

4240

In [129]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.3 Se filtran skus especificos a nivel compañia-sucursal-sku (SE NECESITA EL INPUT DE NICARAGUA)

In [130]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [131]:
pan_rec.id_cliente.nunique()

4240

In [132]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [133]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [134]:
test

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal
0,CAM|0081|120250,523192,0081,05
1,CAM|0081|121190,500094,0081,05
2,CAM|0081|118300,517499,0081,05
3,CAM|0081|118120,517507,0081,05
4,CAM|0081|119020,509552,0081,05
...,...,...,...,...
111963,CAM|0081|101205,523983,0081,02
111964,CAM|0081|93023,522501,0081,02
111965,CAM|0081|101205,523648,0081,02
111966,CAM|0081|101189,511908,0081,02


In [135]:
tz = pytz.timezone("America/Lima")
fecha_manana = (datetime.now(tz) + timedelta(days=1)).strftime("%d_%m_%Y")

bucket = "aje-dl-prod-us-east-2-399723489351-external-data"
prefix = "aje/analiticaAvanzada/gt/PS_Carga_SKU_"
base_path = f"s3://{bucket}/"

print("fecha_manana:",fecha_manana)

fecha_manana: 31_03_2026


In [136]:
def cargar_sku():
    # 1. Intentar carga directa de mañana
    path_manana = f"{base_path}{prefix}{fecha_manana}.xlsx"
    try:
        df = pd.read_excel(path_manana, sheet_name="Hoja1")
        print(f"Cargado archivo de mañana: {fecha_manana}")
        return df
    except Exception:
        print(f"No se encontró archivo para {fecha_manana}. Buscando el más reciente...")

    # 2. Buscar el último archivo disponible con el patrón
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    if 'Contents' in response:
        # Filtrar solo archivos .xlsx y ordenar por fecha de modificación
        archivos = [obj for obj in response['Contents'] if obj['Key'].endswith('.xlsx')]
        ultimo_archivo = sorted(archivos, key=lambda x: x['LastModified'], reverse=True)[0]
        
        path_reciente = f"s3://{bucket}/{ultimo_archivo['Key']}"
        df = pd.read_excel(path_reciente, sheet_name="Hoja1")
        print(f"Cargado archivo más reciente: {ultimo_archivo['Key']}")
        return df
    else:
        raise FileNotFoundError("No se encontraron archivos con el patrón PS_Carga_SKU_")

In [137]:
sku_disponible = cargar_sku()

Cargado archivo de mañana: 31_03_2026


In [138]:
# base_path = (
#     "s3://aje-dl-prod-us-east-2-399723489351-external-data/"
#     "aje/analiticaAvanzada/gt/"
# )

# try:
#     # Intentar leer el archivo de mañana
#     sku_disponible = pd.read_excel(
#         f"{base_path}PS_Carga_SKU_{fecha_manana}.xlsx",
#         sheet_name="Hoja1"
#     )
#     fecha_usada = fecha_manana

# except FileNotFoundError:
#     # Si no existe, leer el archivo de hoy
#     sku_disponible = pd.read_excel(
#         f"{base_path}PS_Carga_SKU_{fecha_hoy}.xlsx",
#         sheet_name="Hoja1"
#     )
#     fecha_usada = fecha_hoy
# print(f"Archivo de SKUs cargado para la fecha: {fecha_usada}")

In [139]:
print(sku_disponible.shape)
sku_disponible.head()

(49, 5)


,fecha_carga,cod_pais,cod_compañia,cod_sucursal,cod_producto
0,31-03-2026,8,81,2,500032
1,31-03-2026,8,81,2,500094
2,31-03-2026,8,81,2,500138
3,31-03-2026,8,81,2,500862
4,31-03-2026,8,81,2,509552


In [140]:
sku_disponible.rename(columns={'cod_compañia': 'cod_compania', 'cod_producto': 'cod_articulo_magic'}, inplace=True)

In [141]:
sku_disponible["cod_compania"] = (sku_disponible["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
sku_disponible["cod_sucursal"] = (sku_disponible["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [142]:
sku_disponible.cod_articulo_magic.nunique()

35

In [143]:
test = pd.merge(test, sku_disponible[["cod_compania","cod_sucursal","cod_articulo_magic"]].drop_duplicates(), on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [144]:
# Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
test.shape

(28351, 4)

In [145]:
test.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal
0,CAM|0081|121190,500094,0081,05
1,CAM|0081|118300,517499,0081,05
2,CAM|0081|118120,517507,0081,05
3,CAM|0081|119020,509552,0081,05
4,CAM|0081|117770,509552,0081,05


In [146]:
pan_rec = test[["id_cliente","cod_articulo_magic"]].copy()

In [147]:
pan_rec.id_cliente.nunique()

4214

In [148]:
PS_VAL_promedio_recs_por_cliente_stock = (
    pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

6.72781205505458

In [149]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.4 Recomendar de acuerdo a Subida, Bajada, Mantener

In [150]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [151]:
PS_VAL_promedio_recs_por_cliente_modelo = (
    pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_modelo

6.72781205505458

In [152]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [153]:
pan_rec.shape

(28351, 2)

In [154]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [155]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [156]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [157]:
df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-30 16:51:33.051015-05:00
fecha_30dias_atras 2026-02-28
fecha_60dias_atras 2026-01-29


In [158]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_30324/261865676.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [159]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [160]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,34208,500032,29035.9194,37383.7469,-22.330098,-22.330098,B,2
1,34208,500094,25784.4980,10313.7993,149.999998,149.999998,S,0
2,34208,500138,70775.0440,72691.8688,-2.636918,-2.636918,B,2
3,34208,500179,3871.4559,3387.5240,14.285711,14.285711,S,0
4,34208,500286,21459.3564,28113.4204,-23.668639,-23.668639,B,2
...,...,...,...,...,...,...,...,...
4768,34287,598821,8615.8738,6940.5646,24.137938,24.137938,S,0
4769,34287,598822,1076.9849,1076.9848,0.000009,0.000009,S,0
4770,34287,598823,1196.6497,598.3246,100.000084,100.000084,S,0
4771,34287,599949,0.0000,239.3298,-100.000000,-100.000000,B,2


In [161]:
pan_rec.shape

(28351, 3)

In [162]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [163]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [164]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [165]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [166]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
12464,CAM|0081|100001,523642,34260,0.0,12464
12475,CAM|0081|100001,523645,34260,1.0,12475
12396,CAM|0081|100001,509906,34260,2.0,12396
12491,CAM|0081|100001,522450,34260,2.0,12491
20477,CAM|0081|100005,509906,34256,0.0,20477


In [167]:
pan_rec.shape

(28351, 5)

In [168]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [169]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0081|100001,523642
1,CAM|0081|100001,523645
2,CAM|0081|100001,509906
3,CAM|0081|100001,522450
4,CAM|0081|100005,509906


In [170]:
pan_rec.shape

(28351, 2)

In [171]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.5 Priorizar/mover SKUs para asegurar su recomendacion (Solo correr si se tienen SKUs a priorizar)

In [172]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [173]:
# skus_a_priorizar = [500580, 500713, 501022, 501505, 501603, 502501, 522398, 622061]

In [174]:
# pan_rec.shape

In [175]:
# # SKUs priorizados
# temp_1 = pan_rec[pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]
# # SKUs NO priorizados
# temp_2 = pan_rec[~pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]

In [176]:
# pan_rec = pd.concat([temp_1, temp_2], axis=0, ignore_index=True)

In [177]:
# pan_rec.shape

In [178]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.6 Quitar Recomendaciones de SKUS en especifico (no correr sin tener SKUs especificos)

In [179]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [180]:
# 622402 no deja vender por unidades
skus_sin_precio = []

In [181]:
pan_rec = pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(
    drop=True
)

In [182]:
pan_rec.id_cliente.nunique()

4214

In [183]:
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda = (
#     pan_rec.cod_articulo_magic.nunique()
#  )
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda

In [184]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.7 Despriorizar recomendaciones de las últimas 2 semanas (no correr sin historial)

In [185]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [186]:
pan_rec.id_cliente.nunique()

4214

In [187]:
master_prod = pd.read_csv("Input/D_pan_masterProd.csv")

In [188]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [189]:
ruta_archivo = f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

El archivo en la ruta Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv no existe.


In [190]:
# Directorio donde se encuentran los archivos
directorio = "Output/PS_piloto_v1/"

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r"\d{4}-\d{2}-\d{2}"

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith(".csv") and re.match(
        r"^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$", archivo
    ):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [191]:
last_7_days_2 = sorted(fechas)[-14:]
last_7_days_2

['2026-03-14',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19',
 '2026-03-20',
 '2026-03-21',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28',
 '2026-03-30']

In [192]:
if len(last_7_days_2) > 0:
    # Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
    last_7_days_recs = pd.DataFrame()
    for fecha_rec in last_7_days_2:
        df_temp = pd.read_csv(
            f"Output/PS_piloto_v1/D_base_pedidos_{fecha_rec}.csv",
            dtype={"Compania": "str", "Cliente": "str"},
        )
        df_temp["id_cliente"] = "CAM|" + df_temp["Compania"] + "|" + df_temp["Cliente"]
        df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
        last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
        print(f"{fecha_rec} done")
else:
    last_7_days_recs = []

2026-03-14 done
2026-03-16 done
2026-03-17 done
2026-03-18 done
2026-03-19 done
2026-03-20 done
2026-03-21 done
2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done
2026-03-30 done


In [193]:
if len(last_7_days_recs) > 0:

    # Merge para identificar coincidencias
    df_combinado = pd.merge(
        pan_rec,
        last_7_days_recs,
        left_on=["id_cliente", "cod_articulo_magic"],
        right_on=["id_cliente", "Producto"],
        how="left",
        indicator=True,
    )

    # 1. Registros que están solo en df1 (left_only)
    df_unicos = df_combinado[df_combinado["_merge"] == "left_only"][
        ["id_cliente", "cod_articulo_magic"]
    ]

    # 2. Registros que están en ambos (both) → los mandamos AL FINAL
    df_coinciden = df_combinado[df_combinado["_merge"] == "both"][
        ["id_cliente", "cod_articulo_magic"]
    ]

    # 3. Unimos: primero únicos, luego coincidentes
    df_resultado = pd.concat([df_unicos, df_coinciden], ignore_index=True)

else:
    df_resultado = []

In [194]:
if len(df_resultado)>0:
    print(pan_rec.id_cliente.nunique())

4214


In [195]:
if len(df_resultado)>0:
    df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.8 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [196]:
datos_para_salesforce = (
    pan_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

## 5.9. Combinamos recomendaciones con categorias

In [197]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [198]:
pan_rec.id_cliente.nunique()

4214

In [199]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

6.72781205505458

In [200]:
pan_rec.shape

(35467, 2)

In [201]:
pan_rec = pd.merge(pan_rec,cliente_rec_marca,on=["id_cliente","cod_articulo_magic"], how="left")

In [202]:
pan_rec

,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0081|100001,523645,SNACKS
1,CAM|0081|100005,523648,SNACKS
2,CAM|0081|100008,500032,GASEOSAS
3,CAM|0081|100011,523188,CEREALES
4,CAM|0081|100011,523432,SOPAS
...,...,...,...
35462,CAM|0081|99997,509906,JUGOS LIGEROS
35463,CAM|0081|99997,522761,GASEOSAS
35464,CAM|0081|99997,522761,GASEOSAS
35465,CAM|0081|99997,523188,CEREALES


## 5.10 Juntando las Recomendaciones Finales
 - Se recomienda el 1er sku de cada categoria.
 - Nos quedamos con 5 skus.
 - Se ordena de acuerdo al rank de su subgiro

In [203]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = pan_rec.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1)
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5))
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [204]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

In [205]:
final_rec.id_cliente.nunique()

4214

In [206]:
final_rec.groupby("id_cliente").sku.nunique().mean()

4.157095396298054

## 5.11 Filtro de Recomendaciones por Segmento de cliente
- Solo se recorta la cantidad de recomendaciones de acuerdo al segmento

In [207]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [208]:
final_rec.nunique()

id_cliente              4214
marca_rec                 10
sku                       32
len_marca_rec             11
desc_rec                  32
desc_giro                  7
desc_subgiro              18
new_segment                4
dias_de_visita__c          2
periodo_de_visita__c       1
ultima_visita              5
cod_compania               1
cod_sucursal               2
cod_cliente             4214
cod_modulo                75
cod_ruta                  75
peso                      10
marca_rec_rank             5
dtype: int64

In [209]:
PS_VAL_promedio_recs_por_cliente_negocio = (
    final_rec.groupby("id_cliente")["sku"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_negocio

4.157095396298054

In [210]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

/tmp/ipykernel_30324/3325175710.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)


In [211]:
final_rec.groupby("new_segment")["id_cliente"].nunique()

new_segment
BLINDAR         856
DESARROLLAR    1523
MANTENER       1020
OPTIMIZAR       815
Name: id_cliente, dtype: int64

In [212]:
final_rec.nunique()

id_cliente              4214
marca_rec                 10
sku                       32
len_marca_rec             11
desc_rec                  32
desc_giro                  7
desc_subgiro              18
new_segment                4
dias_de_visita__c          2
periodo_de_visita__c       1
ultima_visita              5
cod_compania               1
cod_sucursal               2
cod_cliente             4214
cod_modulo                75
cod_ruta                  75
peso                      10
marca_rec_rank             4
dtype: int64

In [213]:
final_rec.groupby("id_cliente")["sku"].nunique().mean()

2.3756525866160416

## 5.12. Guardando recomendacion para D&A y Comercial

In [214]:
final_rec.id_cliente.nunique()

4214

In [215]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
2;5    10009
2          2
Name: count, dtype: int64

In [216]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

periodo_de_visita__c
F1    10011
Name: count, dtype: int64

### 5.8.1 Para D&A

In [217]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [218]:
PS_VAL_total_clientes_con_visita_no_venta = final_rec.id_cliente.nunique()
PS_VAL_total_clientes_con_visita_no_venta

4214

In [219]:
final_rec["cod_cliente"] = final_rec["id_cliente"].str.split("|").str[-1]

In [220]:
PS_VAL_marcas_recomendadas = final_rec.marca_rec.unique()
PS_VAL_marcas_recomendadas

array(['JUGOS LIGEROS', 'BEBIDAS DEPORTIVAS', 'SNACKS', 'GASEOSAS',
       'CEREALES', 'SOPAS', 'SUERO', 'ENERGIZANTE', 'AGUA', 'ALOE VERA'],
      dtype=object)

In [221]:
final_rec.to_csv(
    f"Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [222]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Guatemala/Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Guatemala/Output/PS_data_piloto_v1/D_pan_recs_data_2026-03-31.csv'],
 'partitions_values': {}}

In [223]:
final_rec["cod_compania"] = final_rec["cod_compania"].astype(int)
final_rec["cod_sucursal"] = final_rec["cod_sucursal"].astype(int)
final_rec["cod_cliente"] = final_rec["cod_cliente"].astype(int)
final_rec["cod_modulo"] = final_rec["cod_modulo"].astype(int)
final_rec["sku"] = final_rec["sku"].astype(int)

### 5.8.2 Para subir a PS

In [224]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce = recomendaciones_para_salesforce.drop_duplicates()
recomendaciones_para_salesforce["Pais"] = "NI"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

In [225]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [226]:
recomendaciones_para_salesforce.dtypes

Pais        object
Compania    object
Sucursal    object
Cliente      int64
Modulo       int64
Producto     int64
Cajas        int64
Unidades     int64
Fecha       object
dtype: object

In [227]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

In [228]:
recomendaciones_para_salesforce.Sucursal.unique()

array(['02', '05'], dtype=object)

In [229]:
recomendaciones_para_salesforce.Compania.unique()

array(['0081'], dtype=object)

In [230]:
recomendaciones_para_salesforce.Cliente.nunique()

4214

In [231]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
0,NI,0081,02,100001,34582,509906,1,0,2026-03-31
1,NI,0081,02,100001,34582,522450,1,0,2026-03-31
2,NI,0081,02,100005,34587,509906,1,0,2026-03-31
3,NI,0081,02,100005,34587,523648,1,0,2026-03-31
4,NI,0081,02,100008,34582,500032,1,0,2026-03-31
...,...,...,...,...,...,...,...,...,...
10006,NI,0081,02,99961,34520,522450,1,0,2026-03-31
10007,NI,0081,02,99961,34520,523645,1,0,2026-03-31
10008,NI,0081,02,99997,34582,500094,1,0,2026-03-31
10009,NI,0081,02,99997,34582,509906,1,0,2026-03-31


In [232]:
recomendaciones_para_salesforce.Cliente.nunique()

4214

In [233]:
PS_VAL_total_final_clientes_recomendados = (
    recomendaciones_para_salesforce.Cliente.nunique()
)
PS_VAL_total_final_clientes_recomendados

4214

In [234]:
PS_VAL_promedio_final_recomendaciones_por_cliente = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()
)
PS_VAL_promedio_final_recomendaciones_por_cliente

2.3756525866160416

In [235]:
recomendaciones_para_salesforce.groupby("Sucursal")["Producto"].nunique()

Sucursal
02    20
05    26
Name: Producto, dtype: int64

In [236]:
PS_VAL_total_final_skus_recomendados = (
    recomendaciones_para_salesforce.Producto.nunique()
)
PS_VAL_total_final_skus_recomendados

32

**Conteo de marcas recomendadas**

In [237]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)

In [238]:
PS_VAL_total_combinacion_rec_por_marca = count.shape[0]
PS_VAL_total_combinacion_rec_por_marca

222

**Conteo de SKUs recomendadas**

In [239]:
count2 = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["Cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["Cliente"] / count2["Cliente"].sum()
count2["fa"] = count2["f"].cumsum()

In [240]:
PS_VAL_total_combinacion_rec_por_sku = count2.shape[0]
PS_VAL_total_combinacion_rec_por_sku

986

In [241]:
PS_VAL_total_combinacion_rec_por_sku_al_80 = count2[count2["fa"] <= 0.8].shape[0]
PS_VAL_total_combinacion_rec_por_sku_al_80

289

In [242]:
recomendaciones_para_salesforce["Cliente"] = recomendaciones_para_salesforce[
    "Cliente"
].astype(str)
rutas_cliente["cod_cliente"] = rutas_cliente["cod_cliente"].astype(str)

In [243]:
PS_VAL_rutas_skus_clientes_por_sucursal = pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)
PS_VAL_rutas_skus_clientes_por_sucursal = (
    PS_VAL_rutas_skus_clientes_por_sucursal.groupby("Sucursal")
    .agg({"cod_ruta": "nunique", "Cliente": "nunique", "Producto": "nunique"})
    .reset_index()
)
PS_VAL_rutas_skus_clientes_por_sucursal.columns = [
    "Sucursal",
    "cant_rutas",
    "cant_clientes",
    "cant_skus",
]
PS_VAL_rutas_skus_clientes_por_sucursal

,Sucursal,cant_rutas,cant_clientes,cant_skus
0,02,64,3349,20
1,05,11,865,26


In [244]:
print(f"RESUMEN PS para {fecha_tomorrow}")
print(f"*" * 20, "Metricas Iniciales", "*" * 20)
print(f"{'PS_VAL_total_clientes_venta:':<70}{PS_VAL_total_clientes_venta}")
print(f"{'PS_VAL_total_clientes_visita:':<70}{PS_VAL_total_clientes_visita}")
print(f"{'PS_VAL_total_clientes_visita_trad:':<70}{PS_VAL_total_clientes_visita_trad}")
print(f"{'PS_VAL_ultima_venta:':<70}{PS_VAL_ultima_venta}")
print(f"{'PS_VAL_ultima_visita:':<70}{PS_VAL_ultima_visita}")
print(f"{'PS_VAL_total_skus_venta:':<70}{PS_VAL_total_skus_venta}")
print(f"*" * 20, "Metricas del Proceso", "*" * 20)
print(f"{'PS_VAL_total_clientes_merged:':<70}{PS_VAL_total_clientes_merged}")
print(f"{'PS_VAL_total_fechas_unicas:':<70}{PS_VAL_total_fechas_unicas}")
print(f"{'PS_VAL_total_skus_merged:':<70}{PS_VAL_total_skus_merged}")
print(
    f"{'PS_VAL_total_clientes_imp_netovta_positivo:':<70}{PS_VAL_total_clientes_imp_netovta_positivo}"
)
print(
    f"{'PS_VAL_total_clientes_para_manhana:':<70}{PS_VAL_total_clientes_para_manhana}"
)
print(
    f"{'PS_VAL_total_skus_venta_para_manhana:':<70}{PS_VAL_total_skus_venta_para_manhana}"
)
# print(f"{'PS_VAL_total_skus_recomendados:':<70}{PS_VAL_total_skus_recomendados}")
# print(
#     f"{'PS_VAL_total_clientes_recomendados:':<70}{PS_VAL_total_clientes_recomendados}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_con_stock:':<70}{PS_VAL_total_skus_recomendados_con_stock}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_y_activos:':<70}{PS_VAL_total_skus_recomendados_stock_y_activos}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda:':<70}{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}"
# )
print(f"*" * 20, "Metricas Finales", "*" * 20)
print(
    f"{'PS_VAL_total_final_skus_recomendados:':<70}{PS_VAL_total_final_skus_recomendados}"
)
print(
    f"{'PS_VAL_total_final_clientes_recomendados:':<70}{PS_VAL_total_final_clientes_recomendados}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_modelo:':<70}{PS_VAL_promedio_recs_por_cliente_modelo}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_stock:':<70}{PS_VAL_promedio_recs_por_cliente_stock}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_negocio:':<70}{PS_VAL_promedio_recs_por_cliente_negocio}"
)
print(
    f"{'PS_VAL_promedio_final_recomendaciones_por_cliente:':<70}{PS_VAL_promedio_final_recomendaciones_por_cliente}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_marca:':<70}{PS_VAL_total_combinacion_rec_por_marca}"
)
print(f"{'PS_VAL_marcas_recomendadas:':<70}{PS_VAL_marcas_recomendadas}")
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku:':<70}{PS_VAL_total_combinacion_rec_por_sku}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku_al_80:':<70}{PS_VAL_total_combinacion_rec_por_sku_al_80}"
)
print(
    "PS_VAL_rutas_skus_clientes_por_sucursal:\n",
    PS_VAL_rutas_skus_clientes_por_sucursal,
)

RESUMEN PS para 2026-03-31
******************** Metricas Iniciales ********************
PS_VAL_total_clientes_venta:                                          13270
PS_VAL_total_clientes_visita:                                         12994
PS_VAL_total_clientes_visita_trad:                                    12994
PS_VAL_ultima_venta:                                                  ['2026-03-29']
PS_VAL_ultima_visita:                                                 ['2026-03-28']
PS_VAL_total_skus_venta:                                              87
******************** Metricas del Proceso ********************
PS_VAL_total_clientes_merged:                                         12948
PS_VAL_total_fechas_unicas:                                           332
PS_VAL_total_skus_merged:                                             87
PS_VAL_total_clientes_imp_netovta_positivo:                           12948
PS_VAL_total_clientes_para_manhana:                                   4258
PS_V

In [245]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

2.3756525866160416

In [246]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().value_counts()

Producto
3    1467
2    1216
1     982
4     549
Name: count, dtype: int64

In [247]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [248]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Nicaragua/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Nicaragua/Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv'],
 'partitions_values': {}}

In [249]:
recomendaciones_para_salesforce.shape

(10011, 9)

In [250]:
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["cod_cliente"] = (
    pan_ventas["cod_cliente"].astype(str)
)

In [251]:
val = pd.merge(recomendaciones_para_salesforce,pan_prod,left_on="Producto",right_on="cod_articulo_magic",how="left")
val = pd.merge(val,pan_ventas[["cod_compania","cod_cliente","cod_ruta","desc_sucursal","fecha_proceso"]].drop_duplicates(),left_on=["Compania","Cliente"],right_on=["cod_compania","cod_cliente"],how="left")

In [252]:
val

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_articulo_magic,desc_articulo,cod_compania,cod_cliente,cod_ruta,desc_sucursal,fecha_proceso
0,NI,0081,02,100001,34582,509906,1,0,2026-03-31,509906,CIFRUT FRUIT PET NO RETORNABLE 2000 ml 6 pack,0081,100001,34260,CEDIS XOLOTLÁN,20260330
1,NI,0081,02,100001,34582,522450,1,0,2026-03-31,522450,SPORADE FRUIT PUNCH PET NO RETORNABLE 360 ML 12,0081,100001,34260,CEDIS XOLOTLÁN,20260330
2,NI,0081,02,100005,34587,509906,1,0,2026-03-31,509906,CIFRUT FRUIT PET NO RETORNABLE 2000 ml 6 pack,0081,100005,34256,CEDIS XOLOTLÁN,20260330
3,NI,0081,02,100005,34587,523648,1,0,2026-03-31,523648,D GUSSTO SNACKS NACHOS FRITOS BBQ BOLSA 20 GRS 12,0081,100005,34256,CEDIS XOLOTLÁN,20260330
4,NI,0081,02,100008,34582,500032,1,0,2026-03-31,500032,BIG COLA PET NO RETORNABLE 360 ml 24 pack,0081,100008,34260,CEDIS XOLOTLÁN,20260330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10006,NI,0081,02,99961,34520,522450,1,0,2026-03-31,522450,SPORADE FRUIT PUNCH PET NO RETORNABLE 360 ML 12,0081,99961,34232,CEDIS XOLOTLÁN,20260330
10007,NI,0081,02,99961,34520,523645,1,0,2026-03-31,523645,D GUSSTO SNACKS NACHOS FRITOS QUESO BOLSA 20 G...,0081,99961,34232,CEDIS XOLOTLÁN,20260330
10008,NI,0081,02,99997,34582,500094,1,0,2026-03-31,500094,BIG COLA PET NO RETORNABLE 1000 ML 12,0081,99997,34260,CEDIS XOLOTLÁN,20260330
10009,NI,0081,02,99997,34582,509906,1,0,2026-03-31,509906,CIFRUT FRUIT PET NO RETORNABLE 2000 ml 6 pack,0081,99997,34260,CEDIS XOLOTLÁN,20260330


In [253]:
val.cod_articulo_magic.nunique()

32

In [254]:
val.cod_cliente.nunique()

4214

In [255]:
fecha_hoy_val = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=0)
).strftime("%Y-%m-%d")
fecha_hoy_val

'2026-03-30'

In [256]:
# # Se usa guarda este DF porque contiene info util para comercial
# # Se activó cuando se requerian hacer validaciones
# val.to_csv(f"Val_comercial/D_NI_validacion_recomendaciones_{fecha_hoy_val}.csv",index=False)

### 5.8.3 Enviar Correo con Metricas de PS

In [257]:
# Define la zona horaria de Lima, Perú
lima_tz = pytz.timezone("America/Lima")

# Obtén la fecha y hora actual en la zona horaria de Lima
fecha_hoy_lima = datetime.now(lima_tz)
fecha_hoy = fecha_hoy_lima.strftime("%Y-%m-%d")
fecha_hoy

'2026-03-30'

In [258]:
# Resta un día para obtener la fecha de ayer
fecha_ayer_lima = fecha_hoy_lima - timedelta(days=1)

# Formatea la fecha en el formato año-mes-día
fecha_ayer = fecha_ayer_lima.strftime("%Y-%m-%d")
fecha_ayer

'2026-03-29'

In [259]:
# Obtener el día de la semana (1 es lunes, 7 es domingo)
dia_semana_hoy = (fecha_hoy_lima - timedelta(days=0)).weekday() + 1
dia_semana_hoy

1

In [260]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=1)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_venta[0]:
    mensaje_venta = f"Venta: PS listo. El archivo de ventas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_venta = f"Venta: PS FALTA. Actualizado hasta {PS_VAL_ultima_venta[0]}"

In [261]:
mensaje_venta

'Venta: PS FALTA. Actualizado hasta 2026-03-29'

In [262]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=0)
else:
    fecha_comparacion = fecha_ayer_lima - timedelta(days=-1)

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_visita[0]:
    mensaje_visita = f"Visita: PS listo. El archivo de visitas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_visita = f"Visita: PS FALTA. Actualizado hasta {PS_VAL_ultima_visita[0]}"

In [263]:
mensaje_visita

'Visita: PS FALTA. Actualizado hasta 2026-03-28'

In [264]:
# Credenciales para correo
remitente = "david.porta@ajegroup.com"
password = "pxqn kmuw drlq fvmv"

# Destinatario
asunto = f"Estado y Métricas PS para {fecha_tomorrow}"
# Lista de destinatarios
destinatarios = [
    "david.porta@ajegroup.com",
    "wilmer.rodriguez@ajegroup.com",
    "masaru.gonzales@ajegroup.com",
    "gherald.barzola@ajegroup.com",
    "jorge.delgado.pe@csoluciones.pe",
    "melissa.cotrina@ajegroup.com"
]
#destinatarios = ["david.porta@ajegroup.com"]

# Cuerpo de Correo
msg = MIMEMultipart()
msg["From"] = remitente
msg["To"] = ", ".join(destinatarios)
msg["Subject"] = asunto

In [265]:
# Convertir el DataFrame a HTML si PS_VAL_rutas_skus_clientes_por_sucursal es un DataFrame
tabla_rutas_skus_clientes = PS_VAL_rutas_skus_clientes_por_sucursal.to_html(index=False)

# quitados
# <tr><td>SKUs con stock y activos:</td><td>{PS_VAL_total_skus_recomendados_stock_y_activos}</td></tr>
# <tr><td>SKUs con stock, activos y precio según el lambda:</td><td>{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}</td></tr>
# <tr><td>SKUs con stock:</td><td>{PS_VAL_total_skus_recomendados_con_stock}</td></tr>
# <h4>Métricas luego de correr el Modelo</h4>
# <table>
#     <tr><td>SKUs recomendados:</td><td>{PS_VAL_total_skus_recomendados}</td></tr>
#     <tr><td>Clientes recomendados:</td><td>{PS_VAL_total_clientes_recomendados}</td></tr>      
    
# </table>
# Agregar mensaje al cuerpo del correo en formato HTML
cuerpo_correo = f"""
<html>
<head>
    <style>
        table {{
            width: 50%;
            border-collapse: collapse;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid black;
            text-align: left;
        }}
        th {{
            background-color: #f2f2f2;
        }}
    </style>
</head>
<body>
    <h2>NICARAGUA</h2>
    <h3>Status de subida para {fecha_tomorrow}:</h3>
    <p>{mensaje_venta}</p>
    <p>{mensaje_visita}</p>
    
    <h3>Pendientes:</h3>
    <p style="font-style: italic; color: #555;">Se pide apoyo a Masaru del Equipo Comercial para validar 
    que la subida de PS a Salesforce se realizó de manera satisfactoria.</p>
    
    <h3>Resumen PS Costa Rica para {fecha_tomorrow}:</h3>

    <h4>Métricas de Input</h4>
    <table>
        <tr><td>Clientes en Historico de Ventas:</td><td>{PS_VAL_total_clientes_venta}</td></tr>
        <tr><td>Clientes en archivo de Visitas:</td><td>{PS_VAL_total_clientes_visita}</td></tr>
        <tr><td>Clientes en archivo de Visitas y Canal Tradicional:</td><td>{PS_VAL_total_clientes_visita_trad}</td></tr>
        <tr><td>Fecha ultima venta liquidada:</td><td>{PS_VAL_ultima_venta}</td></tr>
        <tr><td>Fecha ultima visita:</td><td>{PS_VAL_ultima_visita}</td></tr>
        <tr><td>SKUs distintos en Historico de Ventas:</td><td>{PS_VAL_total_skus_venta}</td></tr>
    </table>

    <h4>Métricas del Proceso</h4>
    <table>
        <tr><td>Clientes combinados de ventas y visitas:</td><td>{PS_VAL_total_clientes_merged}</td></tr>
        <tr><td>Fechas de liquidacion unicas en ventas:</td><td>{PS_VAL_total_fechas_unicas}</td></tr>
        <tr><td>SKUs distintos en ventas y visitas:</td><td>{PS_VAL_total_skus_merged}</td></tr>
        <tr><td>Clientes con importe de venta positivo:</td><td>{PS_VAL_total_clientes_imp_netovta_positivo}</td></tr>
        <tr><td>Clientes para mañana:</td><td>{PS_VAL_total_clientes_para_manhana}</td></tr>
        <tr><td>SKUs para clientes de mañana:</td><td>{PS_VAL_total_skus_venta_para_manhana}</td></tr>
    </table>
    
    <h4>Métricas Finales</h4>
    <table>
        <tr><td>SKUs finales recomendados:</td><td>{PS_VAL_total_final_skus_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados:</td><td>{PS_VAL_total_final_clientes_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados + clientes con visita y sin venta:</td><td>{PS_VAL_total_clientes_con_visita_no_venta}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego del modelo:</td><td>{PS_VAL_promedio_recs_por_cliente_modelo}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de stock:</td><td>{PS_VAL_promedio_recs_por_cliente_stock}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reglas de negocio:</td><td>{PS_VAL_promedio_recs_por_cliente_negocio}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reducir skus por segmento:</td><td>{PS_VAL_promedio_final_recomendaciones_por_cliente}</td></tr>
        <tr><td>Combinaciones de marca en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_marca}</td></tr>
        <tr><td>Marcas recomendadas:</td><td>{PS_VAL_marcas_recomendadas}</td></tr>
        <tr><td>Combinaciones de SKUs en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_sku}</td></tr>
        <tr><td>Combinaciones de SKUs al 80% de clientes:</td><td>{PS_VAL_total_combinacion_rec_por_sku_al_80}</td></tr>
    </table>

    <h4>Cantidad de Rutas, SKUs y Clientes por Sucursal:</h4>
    {tabla_rutas_skus_clientes}

</body>
</html>
"""

msg.attach(MIMEText(cuerpo_correo, "html"))

In [266]:
# Login SMTP de Gmail
server = smtplib.SMTP("smtp.gmail.com", 587)
server.starttls()
server.login(remitente, password)

(235, b'2.7.0 Accepted')

In [267]:
# Enviar correo
try:
    server.sendmail(remitente, destinatarios, msg.as_string())
    server.quit()
    print("Correo enviado")
except smtplib.SMTPException as e:
    print(f"Error SMTP al enviar correo: {e}")
except Exception as e:
    print(f"Error general al enviar correo: {e}")
except:
    print("Error al enviar correo")

Correo enviado


In [268]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [269]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel

{'status': 'ok', 'restart': False}

In [270]:
#